In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Downloading E-mini S&P 500 futures
es = yf.download(
    "ES=F",
    start="2025-07-30",
    end="2026-03-10",
    interval="60m",
)

es = es.reset_index()
es['Datetime'] = pd.to_datetime(es['Datetime'])
es = es.set_index('Datetime')

overnight = es[
    (es.index.time >= pd.to_datetime("18:00").time()) |
    (es.index.time <= pd.to_datetime("09:29").time())
]

overnight['Return'] = overnight['Close'].pct_change()
overnight['Date'] = overnight.index.date

overnight_features = overnight.groupby('Date').agg(
    Overnight_Return = ('Return', 'sum'),
    Overnight_Volatility = ('Return', 'std'),
    Futures_Last_Price = ('Close', 'last')
)

overnight_features.index = pd.to_datetime(overnight_features.index)

overnight_features.to_csv("ES_overnight_features.csv")
overnight_features.to_parquet("ES_overnight_features.parquet")